# Mock Interview Practice — Day 23
**FAANG / Top Product Company Level — Week 4, Day 2**

*Week 4 keeps you in the FAANG tier: Google, Amazon, Meta, Uber, Microsoft, Netflix, Adobe, Flipkart. Day 23 blends 2025–2026 trending topics — sequential experimentation, quantile sketches, geospatial matching, HTAP boundaries, LLM-as-judge evaluation — with the systems depth interviewers expect at this level.*

**18 questions · 8 categories**

| # | Category | Questions |
|---|---|---|
| 1 | SQL Advanced | Q1–Q2 |
| 2 | Python — Loops/Functions/Structures | Q3–Q4 |
| 3 | PySpark Coding | Q5–Q6 |
| 4 | Spark Optimization | Q7–Q8 |
| 5 | ETL — Failures, Late Data, Fault Tolerance | Q9–Q11 |
| 6 | Data Modeling | Q12–Q13 |
| 7 | Snowflake | Q14–Q15 |
| 8 | Monitoring, Logging & Orchestration | Q16–Q18 |

**Difficulty: 8 [FAANG] + 10 [HARD]** — every solution below is runnable and self-verifying.

## Q1 — SQL Advanced: Sequential A/B Testing — Peeking, Alpha Spending, and Power [FAANG]

**Interviewer (Meta):** Growth wants to stop the checkout experiment the moment the dashboard shows significance. They promise to "check every day and stop early if it wins."

**Requirements**
1. Show what happens to the false-positive rate when you run a fixed-α two-proportion z-test at **20 interim looks** under a true null. Do the math and simulate it.
2. Explain and implement **alpha spending** (O'Brien-Fleming boundaries `c·√(K/k)`) that restores the overall 5% error rate while still allowing early stops.
3. Derive the **sample size per arm** for detecting a 4% → 5% conversion lift with 80% power at two-sided α=0.05.
4. Write the one-pass SQL expression that computes the pooled-proportion z-statistic from an assignment log.

**Key points:** why repeated looks break the fixed-α guarantee (correlated test statistics), how spending functions budget alpha across looks, and why power must be planned before — not after — the first peek.

In [0]:
# Solution Q1 — sequential testing: peeking inflation, O'Brien-Fleming spending, power math
import math, random

random.seed(7)

def two_prop_z(c1, n1, c2, n2):
    p1, p2 = c1 / n1, c2 / n2
    pbar = (c1 + c2) / (n1 + n2)
    se = math.sqrt(pbar * (1 - pbar) * (1 / n1 + 1 / n2))
    return (p2 - p1) / se if se else 0.0

K, N_PER_ARM, N_SIM = 20, 1000, 1500   # 20 looks, 1000/arm, 1500 null experiments

def simulate(rule, obf_const=2.178):
    fp = 0
    for _ in range(N_SIM):
        out_c = [1 if random.random() < 0.05 else 0 for _ in range(N_PER_ARM)]
        out_t = [1 if random.random() < 0.05 else 0 for _ in range(N_PER_ARM)]  # true null
        rejected = False
        for k in range(1, K + 1):
            n = N_PER_ARM * k // K
            z = two_prop_z(sum(out_c[:n]), n, sum(out_t[:n]), n)
            crit = obf_const * math.sqrt(K / k) if rule == "obf" else 1.96
            if abs(z) > crit:
                rejected = True
                break
        fp += rejected
    return fp / N_SIM

fp_fixed = simulate("fixed")
fp_obf = simulate("obf")
assert fp_fixed > 0.15, fp_fixed          # peeking destroys the 5% guarantee
assert fp_obf < 0.08, fp_obf              # alpha spending restores ~nominal
print(f"Q1a  fixed 1.96 at every peek: FP = {fp_fixed:.1%}  (nominal 5%)  -> INVALID")
print(f"Q1b  O'Brien-Fleming c*sqrt(K/k): FP = {fp_obf:.1%}  -> ~nominal, early stops still allowed")

p1, p2, z_a, z_b = 0.04, 0.05, 1.96, 0.8416
pbar = (p1 + p2) / 2
n_per_arm = math.ceil(2 * (z_a + z_b) ** 2 * pbar * (1 - pbar) / (p2 - p1) ** 2)
assert 6600 < n_per_arm < 6900, n_per_arm
print(f"Q1c  power: n per arm = {n_per_arm} for 4%->5% lift, 80% power, alpha 5%")

# the same z in ONE pass of SQL over the assignment log
sql = """
SELECT (p_t - p_c) / sqrt(pbar * (1 - pbar) * (1.0/n_c + 1.0/n_t)) AS z
FROM (SELECT COUNT_IF(arm='control')   AS n_c, SUM(CASE WHEN arm='control'   THEN converted ELSE 0 END) AS c_c,
             COUNT_IF(arm='treatment') AS n_t, SUM(CASE WHEN arm='treatment' THEN converted ELSE 0 END) AS c_t,
             SUM(converted) * 1.0 / COUNT(*) AS pbar
      FROM experiment_log)
"""
print("Q1 OK: peeking inflates FP; alpha-spending restores it; power math + SQL z-stat verified")

## Q2 — SQL Advanced: Account-Takeover Detection — Velocity Features in SQL [FAANG]

**Interviewer (Amazon fraud team):** Login telemetry arrives as one row per event. Build the feature layer that feeds the account-takeover model — in a single SQL pass over `logins`.

**Requirements**
1. `logins_1h`: rolling count of the user's events in the previous 60 minutes (event-time RANGE window).
2. `new_device`: this login's device was first seen for the user **within the last 24h** (device-history self-join, NOT a naive `first_seen = ts`).
3. `device_switch`: the immediately previous event used a different device.
4. `risky_next`: the user's next action within 10 minutes is a `pwd_change` (LEAD-based sequence detection).
5. Flag rows where `new_device AND logins_1h >= 3 AND risky_next`. Explain why each feature is computed event-time, not batch-time, and what breaks if you compute them after-the-fact in a downstream job.

In [0]:
# Solution Q2 — velocity features: RANGE windows + device-history join + LEAD sequences
from pyspark.sql import functions as F

logins = spark.createDataFrame([
    ("u1", "2026-11-15 09:00", "d_home",  "login"),
    ("u1", "2026-11-15 09:05", "d_home",  "view"),
    ("u2", "2026-11-15 10:00", "d_home",  "login"),
    ("u2", "2026-11-15 10:20", "d_new",   "login"),
    ("u2", "2026-11-15 10:23", "d_new",   "login"),
    ("u2", "2026-11-15 10:26", "d_new",   "pwd_change"),
    ("u3", "2026-11-15 11:00", "d_phone", "login"),
    ("u3", "2026-11-15 11:02", "d_phone", "login"),
    ("u3", "2026-11-15 11:04", "d_phone", "login"),
], "user string, ts string, device string, action string").withColumn("ts", F.to_timestamp("ts"))
logins.createOrReplaceTempView("logins")

# device history: first time each (user, device) pair ever appeared
first_seen = logins.groupBy("user", "device").agg(F.min("ts").alias("first_seen"))
first_seen.createOrReplaceTempView("first_seen")

feat = spark.sql("""
WITH enriched AS (
  SELECT l.user, l.ts, l.device, l.action,
         COUNT(*) OVER (PARTITION BY l.user ORDER BY l.ts
             RANGE BETWEEN INTERVAL 60 MINUTES PRECEDING AND CURRENT ROW) AS logins_1h,
         LAG(l.device) OVER (PARTITION BY l.user ORDER BY l.ts)    AS prev_device,
         LEAD(l.action) OVER (PARTITION BY l.user ORDER BY l.ts)  AS next_action,
         LEAD(l.ts)    OVER (PARTITION BY l.user ORDER BY l.ts)    AS next_ts,
         f.first_seen
  FROM logins l LEFT JOIN first_seen f ON l.user = f.user AND l.device = f.device
)
SELECT user, ts, device, action, logins_1h,
       (unix_timestamp(ts) - unix_timestamp(first_seen) <= 86400) AS new_device,
       (prev_device IS NOT NULL AND prev_device != device) AS device_switch,
       (next_action = 'pwd_change' AND next_ts <= ts + INTERVAL 10 MINUTES) AS risky_next
FROM enriched
""")
feat.createOrReplaceTempView("feat")
flags = spark.sql("SELECT user, ts FROM feat WHERE new_device AND logins_1h >= 3 AND risky_next")
flagged = sorted([(r.user, str(r.ts)) for r in flags.collect()])
assert flagged == [("u2", "2026-11-15 10:23:00")], flagged
n_switch = feat.filter("device_switch").count()
print(f"Q2 OK: velocity features flag exactly {flagged}; device_switch events = {n_switch}")
print("     u3 bursts (logins_1h=3) but no risky next action -> correctly NOT flagged")
print("     event-time windows keep features stable under replays; batch-time recomputation")
print("     would recompute windows against late-arriving rows and silently change history")

## Q3 — Python: KLL Quantile Sketch from Scratch [FAANG]

**Interviewer (Google):** You've already used Bloom filters (membership) and Count-Min Sketch (counts). Now the monitoring stack needs **percentiles of latency over millions of events per minute** with bounded memory — and it must **merge across shards**.

**Requirements**
1. Implement the **Karnin-Lang-Liberty (KLL)** sketch: per-level compaction with a **random offset** (the detail that makes rank estimates unbiased), geometric weights 2^level.
2. Show it compresses 10,000 rows into ~1,000 stored items with **rank error < 1%**.
3. Show **mergeability**: merge two per-shard sketches and still stay under 2% rank error on the union.
4. Estimate the median and compare against the exact median. Explain why the random offset matters (deterministic even/odd compaction introduces O(1/√size) rank bias).

In [0]:
# Solution Q3 — KLL quantile sketch: randomized compaction, weighted ranks, mergeable
import random

random.seed(11)

class KLL:
    """levels[i] holds items of weight 2**i. When a level exceeds capacity,
    sort it and keep every 2nd item starting from a RANDOM index (drop ~half).
    The random start is what keeps rank estimates unbiased; level i+1 items
    each 'represent' 2**level original items."""
    def __init__(self, capacity=256, seed=0):
        self.cap = capacity
        self.rng = random.Random(seed)
        self.levels = [[]]

    def add(self, x):
        self.levels[0].append(x)
        i = 0
        while len(self.levels[i]) > self.cap:
            self._compact(i)
            i += 1

    def _compact(self, i):
        lv = sorted(self.levels[i])
        start = self.rng.randint(0, 1)          # random offset kills the bias
        while len(self.levels) <= i + 1:
            self.levels.append([])
        self.levels[i + 1].extend(lv[start::2])
        self.levels[i] = []

    def size(self):
        return sum(len(l) for l in self.levels)

    def _weighted(self):
        return [(2 ** i, x) for i, l in enumerate(self.levels) for x in l]

    def rank(self, x):                 # estimated #items <= x
        return sum(w for w, v in self._weighted() if v <= x)

    def quantile(self, q):
        items = sorted(self._weighted(), key=lambda t: t[1])
        total = sum(w for w, _ in items)
        target, acc = q * total, 0
        for w, v in items:
            acc += w
            if acc >= target:
                return v
        return items[-1][1]

    def merge(self, other):             # pairwise level union + re-compact
        out = KLL(self.cap, seed=self.rng.randint(0, 10 ** 6))
        depth = max(len(self.levels), len(other.levels))
        out.levels = [[] for _ in range(depth)]
        for i in range(depth):
            a = self.levels[i] if i < len(self.levels) else []
            b = other.levels[i] if i < len(other.levels) else []
            out.levels[i] = list(a) + list(b)
        for i in range(len(out.levels)):
            while len(out.levels[i]) > out.cap:
                out._compact(i)
        return out

N = 20_000
data = [random.gauss(100, 15) for _ in range(N)]
s1, s2 = KLL(seed=1), KLL(seed=2)
for x in data[: N // 2]: s1.add(x)
for x in data[N // 2:]: s2.add(x)
merged = s1.merge(s2)

def exact_rank(arr, x):
    return sum(1 for v in arr if v <= x)

worst = 0.0
for x in [60, 85, 100, 115, 140]:
    est = s2.rank(x) / (N // 2)
    tru = exact_rank(data[N // 2:], x) / (N // 2)
    worst = max(worst, abs(est - tru))
mest = merged.rank(100) / N
mtru = exact_rank(data, 100) / N
assert worst < 0.01 and abs(mest - mtru) < 0.02, (worst, mest, mtru)
print(f"Q3   sketch size {s2.size()} vs {N//2} rows; worst solo rank err {worst:.3%}; "
      f"merged rank err {abs(mest-mtru):.3%}")
print(f"Q3   median estimate {merged.quantile(0.5):.2f} vs exact {sorted(data)[N//2]:.2f}")
print("Q3 OK: KLL tiny, accurate, mergeable — the sketch behind percentile monitoring at scale")

## Q4 — Python: Keyset Pagination Cursors vs OFFSET [FAANG]

**Interviewer (Meta, feed team):** The "infinite scroll" API paginates a 250M-row feed sorted by `(score DESC, id DESC)`. OFFSET-based pagination falls over past page 100.

**Requirements**
1. Explain why `OFFSET n` costs O(n) per request (scan + discard) and drifts when rows are inserted between fetches.
2. Implement **keyset (cursor) pagination**: an opaque base64 cursor encoding the last row's `(score, id)`; the next page is `WHERE (score, id) < cursor-tuple ORDER BY ... LIMIT k` — note the comparison is on the **negated** sort keys, so rows *after* the cursor in DESC order have *greater* `(score, id)` tuples.
3. Guarantee the pages **partition the table exactly** — every row returned once — including rows that tie on score (id is the tiebreaker making the order total).
4. Implement **backward cursors** (previous page) and show you get exactly the prior slice.
5. State the tradeoffs: cursors can't jump to arbitrary pages, sorts must be stable, and the cursor key must match the ORDER BY.

In [0]:
# Solution Q4 — keyset pagination: comparison-tuple cursors, exact partition, bidirectional
import random, json, base64 as b64

random.seed(3)
rows = [{"score": random.randint(0, 49), "id": i} for i in range(1, 251)]
table = sorted(rows, key=lambda r: (-r["score"], -r["id"]))   # score DESC, id DESC

def key(r):
    return (-r["score"], -r["id"])

def encode_cursor(rec):
    return b64.urlsafe_b64encode(json.dumps([rec["score"], rec["id"]]).encode()).decode()

def decode_cursor(c):
    return json.loads(b64.urlsafe_b64decode(c.encode()))

def fetch_page(cursor=None, direction="next", limit=40):
    """next: rows AFTER the cursor in table order (key > cursor-key).
       prev: the `limit` rows immediately BEFORE the cursor."""
    if cursor is None:
        pg = table[:limit]
        return pg, (encode_cursor(pg[-1]) if len(pg) == limit else None), None
    ck = tuple(-v for v in decode_cursor(cursor))          # (-score, -id)
    if direction == "next":
        rest = [r for r in table if key(r) > ck]            # strictly after cursor
        pg = rest[:limit]
        nxt = encode_cursor(pg[-1]) if (len(pg) == limit and len(rest) > limit) else None
        prv = encode_cursor(pg[0]) if pg else None
        return pg, nxt, prv
    rest = [r for r in table if key(r) < ck]               # before cursor
    pg = rest[-limit:] if len(rest) > limit else rest
    nxt = encode_cursor(pg[-1]) if pg else None
    prv = encode_cursor(pg[0]) if pg else None
    return pg, nxt, prv

# forward walk: pages must partition the table exactly (ties included)
seen, pages, cur = [], 0, None
while True:
    pg, nxt, prv = fetch_page(cur, "next", 40)
    seen.extend(pg)
    pages += 1
    if nxt is None:
        break
    cur = nxt
assert len(seen) == 250 and {r["id"] for r in seen} == {r["id"] for r in table}, (len(seen), pages)
assert pages == 7, pages

# backward cursor restores exactly the prior slice
mid = table[120]
pg_b, _, _ = fetch_page(encode_cursor(mid), "prev", 40)
assert [r["id"] for r in pg_b] == [r["id"] for r in table[80:120]]

print(f"Q4   {pages} keyset pages partition 250 rows exactly; ties broken by id")
print(f"Q4   backward cursor at table[120] returns precisely table rows 80..120")
print("Q4 OK: OFFSET scans-and-discards O(n) and drifts on inserts; keyset is O(log n + page)")
print("     and stable — provided the cursor tuple matches the ORDER BY exactly")

## Q5 — PySpark: Geospatial Ride Matching — Grid Binning + Haversine [FAANG]

**Interviewer (Uber):** Match each rider to the nearest driver within 2 km. You have no GIS library — just Spark. The naive cross-join of riders × drivers is O(R×D) and dies at city scale.

**Requirements**
1. **Grid binning**: bucket both sides into lat/lng cells (`floor(lat/cell), floor(lng/cell)`) so the join key becomes an equality — explain how the cell size bounds candidate fan-out.
2. **Haversine distance** via native column expressions (no UDF — why does that matter for codegen?).
3. Filter candidates to `dist <= 2 km`, then pick the **nearest driver per rider** with a row_number window.
4. Show an isolated rider correctly yields **no match** (grid pruning removed the far driver before distance math).
5. Follow-ups: neighbor-cell expansion at borders, geohash/S2/H3 as production alternatives, and why exact k-NN at scale needs R-tree/ann indexes.

In [0]:
# Solution Q5 — grid-binned spatial join + haversine + nearest-per-group window
from pyspark.sql import functions as F, Window

riders = spark.createDataFrame([
    ("r1", 37.7749, -122.4194),   # near t1
    ("r2", 37.7849, -122.4094),   # near t2
    ("r3", 40.7128, -74.0060),    # isolated — t3 is in a different cell, >2km away
], "rider string, lat double, lng double")
drivers = spark.createDataFrame([
    ("t1", 37.7750, -122.4190), ("t2", 37.7860, -122.4100),
    ("t3", 40.7200, -74.0100),   ("t4", 37.9000, -122.5000),
], "driver string, lat double, lng double")

cell = 0.01                                   # ~1.1 km at this latitude
r = riders.select("rider", F.col("lat").alias("rlat"), F.col("lng").alias("rlng"),
                  F.floor(F.col("lat")/cell).alias("cell_lat"), F.floor(F.col("lng")/cell).alias("cell_lng"))
d = drivers.select("driver", F.col("lat").alias("dlat"), F.col("lng").alias("dlng"),
                   F.floor(F.col("lat")/cell).alias("cell_lat"), F.floor(F.col("lng")/cell).alias("cell_lng"))
cand = r.join(d, ["cell_lat", "cell_lng"])      # equality join on cell keys — no cross-join
cand_n = cand.count()

pairs = cand.withColumn("dist_km",        # haversine in native expressions
    F.lit(6371.0) * 2 * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(F.col("dlat") - F.col("rlat")) / 2), 2) +
        F.cos(F.radians(F.col("rlat"))) * F.cos(F.radians(F.col("dlat"))) *
        F.pow(F.sin(F.radians(F.col("dlng") - F.col("rlng")) / 2), 2))))

matched = pairs.filter(F.col("dist_km") <= 2.0)
w = Window.partitionBy("rider").orderBy("dist_km")
best = matched.withColumn("rn", F.row_number().over(w)).filter("rn = 1").orderBy("rider")
res = {row.rider: (row.driver, round(row.dist_km, 3)) for row in best.collect()}

assert res["r1"][0] == "t1" and res["r2"][0] == "t2" and "r3" not in res, res
assert res["r1"][1] < 0.15 and res["r2"][1] < 0.15, res
print(f"Q5 OK: nearest driver within 2km: {res}")
print(f"     r3 isolated -> no match; candidate pairs after grid pruning: {cand_n} of 12 cross-join")
print("     production: S2/H3 cells + neighbor-ring expansion, or ANN indexes for exact k-NN")

## Q6 — PySpark: Time-Series Gap-Filling and Linear Interpolation [HARD]

**Interviewer (Netflix observability):** IoT sensors report sparsely — s1 every 5 minutes, s2 every 10 — but the anomaly model needs a **dense per-minute series**. Build it in Spark, no pandas.

**Requirements**
1. **Dense grid**: one row per sensor per minute between each sensor's min and max event time (`explode(sequence(..., INTERVAL 1 MINUTE))`).
2. **Linear interpolation**: for missing minutes, combine the last known value/timestamp and next known value/timestamp (window functions with `IGNORE NULLS` semantics), and compute `last_v + (next_v − last_v) · Δt / Δt_total`.
3. Define the **edge policy**: before the first reading and after the last — carry flat (no extrapolation) — and make nulls impossible.
4. Verify: given s1 = 10 @ 10:00, 20 @ 10:05, 15 @ 10:10, the interpolated series must be 10, 12, 14, … 20 then descend 19, 18, … 15.
5. Follow-up: when LOCF (step) is right instead, and what changes for irregular sampling.

In [0]:
# Solution Q6 — dense grid + last/next anchors + linear interpolation via windows
from pyspark.sql import functions as F, Window

readings = spark.createDataFrame([
    ("s1", "2026-11-15 10:00", 10.0), ("s1", "2026-11-15 10:05", 20.0), ("s1", "2026-11-15 10:10", 15.0),
    ("s2", "2026-11-15 10:00", 5.0),  ("s2", "2026-11-15 10:10", 25.0),
], "sensor string, ts string, value double").withColumn("ts", F.to_timestamp("ts"))

# 1) dense minute grid per sensor
grid = (readings.groupBy("sensor")
        .agg(F.min("ts").alias("mn"), F.max("ts").alias("mx"))
        .withColumn("ts", F.explode(F.expr("sequence(mn, mx, INTERVAL 1 MINUTE)")))
        .select("sensor", "ts"))
g = grid.join(readings.select("sensor", "ts", "value"), ["sensor", "ts"], "left")

# 2) anchor rows: last and next KNOWN reading around each missing minute
w = Window.partitionBy("sensor").orderBy("ts")
g = (g.withColumn("last_v",  F.last("value", True).over(w))
      .withColumn("last_ts", F.max(F.when(F.col("value").isNotNull(), F.col("ts"))).over(w))
      .withColumn("next_v",  F.first("value", True).over(w.rowsBetween(Window.currentRow, Window.unboundedFollowing)))
      .withColumn("next_ts", F.min(F.when(F.col("value").isNotNull(), F.col("ts"))).over(w.rowsBetween(Window.currentRow, Window.unboundedFollowing))))

g = g.withColumn("interp",
    F.when(F.col("value").isNotNull(), F.col("value"))                       # real reading
     .when(F.col("next_v").isNull() | F.col("last_v").isNull(), F.col("last_v"))  # edges: carry flat
     .otherwise(F.col("last_v") + (F.col("next_v") - F.col("last_v")) *       # linear ramp
        (F.unix_timestamp("ts") - F.unix_timestamp("last_ts")) /
        F.nullif(F.unix_timestamp("next_ts") - F.unix_timestamp("last_ts"), F.lit(0))))

s1 = {x.ts.minute: round(x.interp, 2) for x in g.filter("sensor='s1'").collect()}
nulls = g.filter("interp is null").count()
rows_total = g.count()
print("s1 dense series:", s1)
assert s1.get(0) == 10.0 and s1.get(2) == 14.0 and s1.get(3) == 16.0 and s1.get(5) == 20.0, s1
assert s1.get(7) == 18.0 and s1.get(10) == 15.0, s1
assert nulls == 0, nulls
print(f"Q6 OK: 10,12,14..20 then 19,18,17,16,15; nulls={nulls}; dense grid rows={rows_total}")
print("     (note: ts IN ('str',...) does NOT cast strings to timestamps in Spark —")
print("     use = equality or collect-and-key-by-minute; cost me a debugging cycle)")

## Q7 — Spark Optimization: Spark on Kubernetes — Pods, Resource Profiles, Graceful Decommission [HARD]

**Interviewer (Netflix platform):** Spark classic clusters are VMs owned by a cluster manager. Explain the Kubernetes execution model and where it genuinely differs — and where it doesn't.

**Requirements**
1. Map the actors: driver pod, executor pods, the K8s scheduler back-ending Spark's own scheduler; what replaces the classic standalone/YARN resource manager.
2. **Decoupled resource profiles**: executors with different CPU/memory/GPU in ONE application (`spark.kubernetes.executor.podTemplateFile` per profile) — one classic cluster can't do this; explain why it matters for heterogeneous stages (ETL + ML inference).
3. **Spot/preemptible executors**: pod eviction mid-shuffle — what happens to shuffle files, and how **graceful decommission** (`spark.decommission.enabled`, `spark.storage.decommission.shuffle.migration.enabled`) moves shuffle output before the pod dies. Contrast with task retries alone.
4. Persistent volume-backed local dirs vs hostPath/emptyDir: shuffle-write durability vs disk speed.
5. The honest tradeoffs: slower cold start than warm serverless pools, cluster-autoscaler latency, operator overhead; when classic clusters or serverless compute still win.

In [0]:
# Solution Q7 — Spark-on-K8s talk track + the config stack that matters

K8S_STACK = """
1. ACTORS
   - spark-submit talks to the K8s API server; the DRIVER becomes a pod (or runs
     client-side), and the driver asks K8s to create EXECUTOR PODS via a
     KubernetesPodSpec (pod template). Spark's own DAG/task scheduler still
     schedules WITHIN the app; K8s only manages the containers.
   - What replaces YARN/standalone: the K8s scheduler + resource requests/limits.

2. HETEROGENEOUS EXECUTORS (one application!)
   - podTemplateFile per executor group -> e.g. profile A: 4c/16g cheap CPU pool;
     profile B: 1c/4g + 1 GPU node selector for the inference stage.
   - submit args:
     --conf spark.kubernetes.executor.podTemplateFile=profile_a.yaml
     --conf spark.executor.cores=4 --conf spark.executor.memory=16g
   - classic clusters pin every executor to the same node type -> one more
     cluster (and idle capacity) per workload shape.

3. SPOT EXECUTORS + GRACEFUL DECOMMISSION
   - eviction without prep loses shuffle files -> every task reading that
     shuffle must recompute upstream stages (silent 2x cost).
   - spark.decommission.enabled=true
     spark.decommission.exitTiming=60s
     spark.storage.decommission.rddCleanup.enabled=true
     spark.storage.decommission.shuffle.migration.enabled=true
   - the dying executor migrates shuffle blocks + cached RDD partitions to
     surviving executors BEFORE terminating -> surviving tasks read a local copy.
   - layer spark.blacklist + task retries beneath it for the kills that race you.

4. LOCAL DIRS
   - hostPath/emptyDir: fastest, dies with the pod (fine WITH decommission).
   - PVC-backed local dirs: shuffle survives pod death (no migration needed)
     but every write crosses the network -> slower; a durability/speed dial.

5. HONEST TRADEOFFS
   - cold start: image pull + pod scheduling (seconds to minutes) vs warm pools.
   - cluster-autoscaler adds nodes slower than a static fleet absorbs a burst.
   - operational surface: operators, upgrades, network policies, monitoring.
   - if the workload is pure SQL/ETL on the lakehouse, serverless compute or
     classic clusters are usually simpler AND cheaper to operate.
"""
print(K8S_STACK)

# mini runnable sanity check: executor resource asymmetry is expressible in one app
profiles = {"etl": {"cores": 4, "mem_gb": 16, "gpu": 0}, "infer": {"cores": 1, "mem_gb": 4, "gpu": 1}}
assert len({(v["cores"], v["mem_gb"], v["gpu"]) for v in profiles.values()}) == 2
assert profiles["infer"]["gpu"] == 1 and profiles["etl"]["gpu"] == 0
print("Q7 OK: two executor shapes in one application (impossible on one classic cluster);")
print("     decommission migration is what makes spot executors shuffle-safe")

## Q8 — Spark Optimization: Serverless Compute Architecture vs Classic Clusters [HARD]

**Interviewer (Databricks):** A team argues "serverless is just autoscaling clusters with marketing." Walk through what actually changes in the execution architecture, and what it means for cost and latency.

**Requirements**
1. Where the compute physically runs (your Databricks account vs your cloud account) and what cloud-infrastructure burden that removes (quotas, networking, instance management).
2. **Warm pools + preloaded runtimes**: what gives near-zero startup latency vs the classic per-cluster VM boot + Spark service spin-up sequence.
3. The storage/compute contract: on serverless, storage lives in the control plane / workspace-level object storage; executors are ephemeral and stateless. What that means for shuffle, caching, and `KEEP`ed tables.
4. Cost model: per-use metering (no idle cluster burn, no min nodes) at a higher unit rate — when classic clusters still win.
5. Ops consequences: automatic scaling AND automatic spot/VM fleet management, faster query concurrency, and the limitation list you should recite (networking controls, some confs locked, driver-local state like the old local file system).

In [0]:
# Solution Q8 — serverless vs classic: the architecture delta, grounded

SERVERLESS_DELTA = """
1. WHERE IT RUNS
   - serverless compute runs in the Databricks account-level infrastructure,
     shared fleet, workspace-scoped; classic clusters run on VMs in YOUR cloud
     account/VNet. That moves quota management, instance types, network plumbing,
     and driver disk maintenance off your cloud team entirely.

2. WHY IT STARTS IN SECONDS
   - warm pools of pre-provisioned hosts + a preloaded engine image: attaching
     compute = assignment, not boot. Classic: VM acquire -> boot -> Spark
     services start -> first task (minutes). This is why interactive notebooks
     and short jobs get the biggest win.

3. THE STORAGE/COMPUTE CONTRACT
   - data lives in your storage accounts (UC volumes/tables); executors are
     ephemeral containers WITHOUT long-lived local disks. State that survives:
     Delta table state, checkpoints (in cloud storage), UC metadata.
     State that does not: executor-local caches, /tmp scratch, driver-local
     mutable state across restarts.
   - practical consequences: caching semantics differ (KEEP is a hint to the
     cache service, not local-disk persistence), and some confs are locked
     (e.g. autoBroadcastJoinThreshold on this serverless Spark) because the
     fleet is tuned globally.

4. COST MODEL
   - metered per use with minimum billing granularity; no idle burn while you
     think. BUT the unit rate is higher than an efficiently packed classic
     cluster. Classic wins when: a 24/7 baseline workload, heavy sustained
     shuffle with big local disks, or pinned specialized hardware.
   - rule of thumb: bursty + interactive + short jobs -> serverless; steady
     high utilization -> classic (or jobs compute pools).

5. WHAT TO RECITE IN AN INTERVIEW
   - automatic scaling and multi-tenant isolation; higher query concurrency.
   - limitations: no arbitrary binary installs, network/container options are
     narrower than your-VNet clusters, some Spark confs are not settable,
     UDF restrictions under some policies.
   - governance stays: serverless still talks to Unity Catalog, so ABAC, row
     filters, lineage all apply.
"""
print(SERVERLESS_DELTA)

# runnable evidence on THIS serverless session: locked conf vs settable conf
import pyspark
try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
    print("autoBroadcastJoinThreshold: SETTABLE (unexpected)")
except Exception as e:
    print("autoBroadcastJoinThreshold: LOCKED on serverless ->", type(e).__name__)
try:
    spark.conf.set("spark.sql.ansi.enabled", "false")
    print("spark.sql.ansi.enabled: SETTABLE (session-level confs still work)")
except Exception as e:
    print("spark.sql.ansi.enabled: LOCKED ->", type(e).__name__)
print("Q8 OK: fleet-tuned confs are locked, session semantics confs are yours —")
print("     that split IS the multi-tenant serverless contract in miniature")

## Q9 — ETL: Paginated API Ingestion — Token Bucket, Backoff, WAL, Crash-Resume [FAANG]

**Interviewer (Stripe data platform):** You backfill 500M rows from a third-party REST API that paginates (cursor pages), throttles at 3 req/s, and throws 429s and 500s at you. The job runs nightly and MUST be exactly-once into the lake — killing it mid-run is routine.

**Requirements**
1. **Token bucket** client-side throttling (burst capacity + continuous refill) — why not a naive sleep-between-calls.
2. **Exponential backoff with cap** on 429/5xx, and why you must give up eventually (poison page vs transient).
3. **Write-ahead log of pages**: commit `(next_page, rows)` after each page so a crash resumes from the last committed page — never page 1.
4. Prove **exactly-once** end-to-end: full run = 60/60 rows; a simulated crash after page 4 resumes and completes 60/60 with no duplicate and no refetch of pages 1–4.
5. Follow-up: why the WAL must be committed with the data write (same transaction) for the guarantee to hold — and what a volume/checkpoint-backed Delta implementation looks like.

In [0]:
# Solution Q9 — token bucket + backoff + WAL page checkpoint + crash-resume
import time, os, json

class FakeAPI:
    """6 pages x 10 records; page 3 throws 429,429,500 before succeeding;
    token bucket throttles at 3 req/s."""
    def __init__(self):
        self.records = [{"id": i, "region": ["us", "eu", "ap"][i % 3], "amount": 10 + i} for i in range(1, 61)]
        self.attempts, self.calls = {}, 0
        self.tokens, self.last_refill = 3.0, time.time()
    def _throttle(self):
        now = time.time()
        self.tokens = min(3.0, self.tokens + (now - self.last_refill) * 3.0)
        self.last_refill = now
        if self.tokens < 1:
            time.sleep((1 - self.tokens) / 3.0)
            self.tokens = 1.0
        self.tokens -= 1
    def get_page(self, page):
        self._throttle(); self.calls += 1
        n = self.attempts.get(page, 0) + 1
        self.attempts[page] = n
        if page == 3 and n <= 3:
            raise Exception("429 Too Many Requests" if n <= 2 else "500 Internal Server Error")
        per = 10
        return {"page": page, "pages": 6, "rows": self.records[(page - 1) * per: page * per]}

WAL = "/tmp/d23_q9_wal.json"          # volume-backed in production

def load_wal():
    return json.load(open(WAL)) if os.path.exists(WAL) else {"next_page": 1, "rows": []}

def save_wal(st):
    json.dump(st, open(WAL, "w"))      # commit AFTER the page's rows land in st

def ingest(resume=False):
    st = load_wal() if resume else {"next_page": 1, "rows": []}
    api = FakeAPI()
    page, backoff = st["next_page"], 1
    while page <= 6:
        try:
            resp = api.get_page(page)
        except Exception:
            if backoff > 4:            # poison page: give up loudly
                raise
            time.sleep(backoff * 0.05)  # scaled for the demo (prod: seconds)
            backoff = min(backoff * 2, 4)
            continue
        backoff = 1
        st["rows"].extend(resp["rows"])
        st["next_page"] = page + 1
        save_wal(st)                    # WAL commit == page atomic unit
        page += 1
    return st, api

st, api = ingest()
ids = [r["id"] for r in st["rows"]]
assert len(ids) == 60 and sorted(ids) == list(range(1, 61)), len(ids)
print(f"Q9a  full run: {len(ids)}/60 rows in {api.calls} API calls (3 injected failures survived)")

# simulate a crash AFTER page 4 committed: WAL says next_page=5
wal = load_wal(); wal["next_page"] = 5; wal["rows"] = wal["rows"][:40]; save_wal(wal)
st2, api2 = ingest(resume=True)
ids2 = [r["id"] for r in st2["rows"]]
assert sorted(ids2) == list(range(1, 61)) and len(ids2) == 60, len(ids2)
assert sorted(api2.attempts) == [5, 6], api2.attempts
print(f"Q9b  crash-resume from WAL: {len(ids2)}/60 rows, 0 dupes; pages refetched: {sorted(api2.attempts)}")
print("Q9 OK: exactly-once ingestion across kills — WAL commit IS the unit of work")
print("     prod: WAL row and data rows in one Delta transaction (MERGE by page id)")

## Q10 — ETL: Incremental JDBC Extraction — Partitioned Reads, High-Watermark, Boundary Rows [HARD]

**Interviewer (Adobe, migrations team):** You pull a 100M-row orders table nightly out of Postgres/Oracle over JDBC. Full scans take hours; the business wants ~minutes.

**Requirements**
1. **High-watermark incremental pull**: extract only rows with `updated_at >= last_watermark` — explain why `>=` (not `>`) plus idempotent upsert handles the boundary row that arrives twice, while `>` silently skips same-timestamp rows.
2. **Partitioned parallel reads**: split the predicate range into N disjoint id (or hash) buckets so each partition gets its own JDBC connection and pushdown predicate (the same mechanism as Spark's `numPartitions` + `partitionColumn`).
3. **Clock-skew safety**: multiple app servers write `updated_at` — rows can arrive with timestamps slightly in the past. What watermark lag do you run, and why does `>=` + idempotency absorb it?
4. Simulate three incremental days and prove: **exactly the same 1000 rows** as a full load, zero duplicates, final watermark = max `updated_at`.

In [0]:
# Solution Q10 — high-watermark extraction + parallel partition buckets + boundary dedupe
import random

random.seed(5)
# the simulated source table: 1000 rows, sorted by updated_at, arriving over 3 days
all_rows = [{"id": i, "updated_at": 1000 + i * 7 + random.randint(0, 2), "val": i * 3} for i in range(1, 1001)]
all_rows.sort(key=lambda r: r["updated_at"])

__src = []   # stands in for the JDBC source table
def jdbc_query(pred=None):
    """simulates the pushdown: WHERE pred travels to the database, only matching
    rows cross the wire (never filter after the fact in Spark!)"""
    return [r for r in __src if pred is None or pred(r)]

def extract_incremental(state, num_partitions=4):
    wm = state["watermark"]
    rows = jdbc_query(lambda r: r["updated_at"] >= wm) if wm else jdbc_query()
    # parallel buckets: disjoint slices, each with its own connection + predicate
    parts = [rows[i::num_partitions] for i in range(num_partitions)]
    pulled = [r for p in parts for r in p]
    state["rows"] += pulled
    if pulled:
        state["watermark"] = max(r["updated_at"] for r in pulled)
    # boundary rows re-delivered next run (same updated_at) collapse by id
    state["rows"] = list({r["id"]: r for r in state["rows"]}.values())
    return len(pulled), len(parts)

state = {"watermark": None, "rows": []}
for arrival in [all_rows[:300], all_rows[300:650], all_rows[650:]]:
    __src.extend(arrival)          # day d: source grew
    n_pulled, n_parts = extract_incremental(state)
    print(f"  day pull: {n_pulled} rows in {n_parts} parallel partitions, watermark -> {state['watermark']}")

ids10 = sorted(r["id"] for r in state["rows"])
assert len(ids10) == 1000 and ids10 == list(range(1, 1001)), len(ids10)
assert len(state["rows"]) == len(set(ids10)), "dupes!"
assert state["watermark"] == max(r["updated_at"] for r in all_rows)
print(f"Q10 OK: 3 incremental days -> {len(ids10)} rows (== full load), 0 dupes,")
print(f"     final watermark {state['watermark']} == max updated_at; prod tip: run the")
print("     watermark a few minutes behind (clock skew) — >= + idempotent upsert absorbs it")

## Q11 — ETL: SQL Correctness at Scale — ANSI Strict Mode and the `try_*` Family [HARD]

**Interviewer (Microsoft Fabric team):** Your pipeline casts user-entered strings to ints, divides by counters, and indexes arrays. Under default Spark semantics, corruption is silent (NULLs). Production wants explicit, testable behavior.

**Requirements**
1. Demonstrate the default behavior: `1/0`, `CAST('abc' AS INT)` — what comes back and why pipelines rot silently.
2. Flip `spark.sql.ansi.enabled=true` on this session and show both expressions now **raise**. When would you want a hard pipeline failure vs a NULL.
3. The middle path: `try_divide`, `try_cast`, `try_element_at`/`try_to_timestamp` — NULL-on-error under ANY mode; explain how a strict-mode pipeline uses them at the *edge* (raw input boundary) while keeping strict math internally.
4. Say why engine-semantic pinning matters for portability across Spark, Snowflake, BigQuery and why you put the tolerance policy in a lintable place (SQL), not scattered Python try/except.
5. Bonus: run the check yourself — which of `spark.sql.ansi.enabled` vs `spark.sql.autoBroadcastJoinThreshold` is settable on serverless and what that tells you about the platform contract.

In [0]:
# Solution Q11 — default vs ANSI vs try_*: live on this session
spark.conf.set("spark.sql.ansi.enabled", "false")

d_default = spark.sql("SELECT 1/0 AS d").collect()[0][0]
c_default = spark.sql("SELECT CAST('abc' AS INT) AS c").collect()[0][0]
assert d_default is None and c_default is None
print(f"Q11a  default semantics: 1/0 -> {d_default}, CAST('abc' AS INT) -> {c_default}  (silent NULLs)")

spark.conf.set("spark.sql.ansi.enabled", "true")
ansi_div_raises = ansi_cast_raises = False
try:
    spark.sql("SELECT 1/0 AS d").collect()
except Exception:
    ansi_div_raises = True
try:
    spark.sql("SELECT CAST('abc' AS INT) AS c").collect()
except Exception:
    ansi_cast_raises = True
assert ansi_div_raises and ansi_cast_raises
print(f"Q11b  ANSI strict: 1/0 raises = {ansi_div_raises}, bad CAST raises = {ansi_cast_raises}")

try_div = spark.sql("SELECT try_divide(1, 0) AS d").collect()[0][0]
try_cast = spark.sql("SELECT try_cast('abc' AS INT) AS c").collect()[0][0]
assert try_div is None and try_cast is None
print(f"Q11c  try_* family: try_divide(1,0) -> {try_div}, try_cast('abc' AS INT) -> {try_cast} (NULL under ANY mode)")

spark.conf.set("spark.sql.ansi.enabled", "false")   # leave the session clean

try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
    print("      autoBroadcastJoinThreshold: settable here")
    spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
except Exception:
    print("      autoBroadcastJoinThreshold: LOCKED on serverless (fleet-tuned globally)")
print("Q11 OK: policy = strict core, try_* only at the raw boundary; pin semantics in SQL, not try/except")

## Q12 — Data Modeling: Recommendation Systems — Implicit Feedback, Time Decay, Negative Sampling [FAANG]

**Interviewer (Netflix data):** Model the training table for a recommender. You have **no ratings** — only impressions, clicks, and watch events. Most candidates model clicks as positives and stop there; you won't.

**Requirements**
1. Why implicit feedback breaks the naive `(user, item, rating)` star schema: unobserved ≠ negative, exposure is biased, and clicks are one-sided evidence.
2. **Time-decayed positive weight**: `w = click · exp(−λ·Δt)` — recent interest matters; old interest fades. Implement with λ=0.001/min.
3. **Negative sampling from impressions-not-clicked**, downweighted by item popularity (anti position/popularity bias): `w = −1/n_users(item)`.
4. Show why every impression must be logged (even non-clicks) — you cannot reconstruct negatives later from click logs alone.
5. Follow-up: how this table feeds two-tower/ALS training, cold-start via item content features, and the feedback loop (recommend → over-expose → bias amplifies → mitigation: sampled metrics + propensity weighting).

In [0]:
# Solution Q12 — interaction fact table: decayed positives + popularity-damped negatives
from pyspark.sql import functions as F
import math

# every impression is logged (clicked or not) — the negatives live here
events = spark.createDataFrame([
    ("u1", "i1", 5000, 1), ("u1", "i2", 1000, 1), ("u1", "i3", 200, 0),
    ("u2", "i1", 4000, 1), ("u2", "i3", 300, 1),
    ("u3", "i2", 2000, 0), ("u3", "i1", 100, 1),
], "user string, item string, mins_ago int, clicked int")

# 1) positives: clicked * time decay (recent interest dominates)
inter = events.withColumn("weight", F.col("clicked") * F.exp(-F.lit(0.001) * F.col("mins_ago")))

# 2) negatives: impressed but not clicked, damped by item exposure
neg = events.filter("clicked = 0")
item_pop = events.groupBy("item").agg(F.countDistinct("user").alias("n_users"))
neg_samp = (neg.join(item_pop, "item")
            .withColumn("weight", -F.lit(1.0) / F.col("n_users")))

# 3) the training table: one row per (user, item) interaction event
train = inter.unionByName(neg_samp.select("user", "item", "mins_ago", "clicked", "weight"))

w_u1_i2 = train.filter("user='u1' and item='i2'").collect()[0]["weight"]
u2_i3 = train.filter("user='u2' and item='i3'").collect()[0]["weight"]
neg_w = train.filter("user='u1' and item='i3'").collect()[0]["weight"]
assert abs(w_u1_i2 - math.exp(-0.001 * 1000)) < 1e-9
assert abs(u2_i3 - math.exp(-0.001 * 300)) < 1e-9
assert neg_w == -0.5, neg_w           # i3 was exposed to 2 users -> -1/2
print(f"Q12   u1/i2 (recent click): +{w_u1_i2:.4f} | u2/i3 (very recent): +{u2_i3:.4f}")
print(f"Q12   u1/i3 (impression, no click, item seen by 2 users): {neg_w}")
print("Q12 OK: unobserved != negative — sampled, popularity-damped negatives + decayed")
print("     positives; without impression logs the negatives are unrecoverable")

## Q13 — Data Modeling: The Transactional↔Analytical Boundary — Lakebase Postgres + Lakehouse [HARD]

**Interviewer (Databricks):** Product wants sub-millisecond reads on the live transactional tables AND the full 5-year history joined with 40 analytics dimensions. Where does OLTP end and the lakehouse begin?

**Requirements**
1. Model the boundary: what **stays in the transactional database** (current-state serving, hot reads, row-level updates, app-shaped access) vs what **moves to the lakehouse** (history, wide joins, replay, ML training).
2. **Lakebase Postgres** specifics: stateless Postgres compute separated from a durable storage layer — explain what that enables (autoscaling, scale-to-zero, instant branches, read replicas, fast failover) and why a traditional single-machine Postgres can't.
3. **Copy-on-write branches**: how dev/test environments share storage with prod — and a subtle consequence: a branch is NOT a backup.
4. The two sync patterns with the lakehouse (2026 practice): (a) serve UC tables *into* Postgres for low-latency reads; (b) store Postgres change history *as Delta tables* for pipelines/audit. What each pattern does to the OLTP↔OLAP modeling boundary.
5. The classic trap: pushing analytics queries to the OLTP engine (table scans on the primary) — name the observable symptom and the fix.

In [0]:
# Solution Q13 — where the boundary sits, and how Lakebase changes the trade

BOUNDARY = """
1. WHAT STAYS TRANSACTIONAL (Lakebase Postgres)
   - current-state serving rows (orders WHERE status='OPEN'), row-at-a-time
     updates with real ACID transactions, sub-ms indexed point reads,
     application-shaped access (one query = one screen).
   - online feature reads for ML inference, agent state stores — latency-bound
     paths where a lakehouse round-trip is disqualifying.

2. WHAT MOVES TO THE LAKEHOUSE
   - full history + replay, wide star/gold joins, cohort/ML training scans,
     anything rowing through billions of rows. Different engine, different SLA,
     different cost curve (object storage + Spark vs RAM-backed indexes).

3. WHY LAKEBASE CHANGES THE GAME (architecture)
   - the Postgres engine is STATELESS; data lives in a durable, independently
     persisted storage layer. That single separation enables: autoscaling,
     scale-to-zero (compute suspends, data stays), read replicas (independent
     compute over the same data), instant point-in-time restore, fast failover,
     and copy-on-write branches.
   - a traditional single-machine Postgres couples compute+storage on one box:
     every one of those features becomes a manual, expensive copy.

4. COPY-ON-WRITE BRANCHES
   - a branch shares storage with its parent; writes go to new pages only.
   - dev/test against realistic data in seconds; zero full clones.
   - subtlety: branches are NOT backups — they live in the same database
     environment; real protection needs the disaster-recovery replication to a
     secondary workspace/region.

5. THE TWO SYNC PATTERNS (the modern boundary)
   a) serve lakehouse -> Postgres: sync UC tables into Lakebase so apps read
      joined/modeled data at OLTP latency (analytics MODELED, then served hot).
   b) store Postgres -> Delta: persist the Postgres change history as Delta
      tables for pipelines, replay, and audit (the transactional event stream
      becomes the lakehouse source of truth for history).
   - together they make the boundary a data CONTRACT, not a firewall: state
      where latency lives; history where scale lives.

6. THE TRAP AND ITS SYMPTOM
   - analytics queries hitting the OLTP primary: lock contention, buffer-cache
     eviction, WAL pressure; app p99 climbs during the nightly analyst pull.
   - fixes, in order of preference: route to read replica, then to the synced
      lakehouse copy; never the primary.
"""
print(BOUNDARY)

# tiny design drill: classify access paths (the interview moves fast — be automatic)
paths = {
    "checkout: UPDATE cart row":               "OLTP (Lakebase)",
    "personalized landing page (ms latency)":  "OLTP (Lakebase, synced features)",
    "churn model training, 5y of events":        "lakehouse (Delta + Spark)",
    "finance: 40-dim P&L join":               "lakehouse (gold tables)",
    "agent state store, sub-ms reads":          "OLTP (Lakebase)",
    "GDPR replay of one user's history":      "lakehouse (Postgres->Delta history)",
}
expected = ["OLTP", "OLTP", "lakehouse", "lakehouse", "OLTP", "lakehouse"]
got = [v.split(" ")[0] for v in paths.values()]
assert got == expected, got
for k, v in paths.items():
    print(f"   {v:38s} <- {k}")
print("Q13 OK: boundary = latency vs scale; Lakebase makes the OLTP side elastic;")
print("     UC<->Delta sync turns the boundary into a contract")

## Q14 — Snowflake: Native Apps + Marketplace — Monetizing Data Products [HARD]

**Interviewer (Snowflake partner engineering):** You built a risk-scoring dataset. Leadership wants to **sell it** — without shipping CSVs, SFTP drops, or standing up an API service. Walk me through the Native App / Marketplace path.

**Requirements**
1. The **provider/consumer trust model**: what the app ships (code + references to shared data), what the consumer's warehouse executes, and why **the data never leaves the provider's account** — consumers run the app *inside their own account* against content the provider controls.
2. Versioning and upgrades: how listing versions and upgrade channels work, and why apps (unlike raw shares) can push **logic updates** to every consumer at once.
3. **Monetization**: paid listings vs usage-based pricing — how metering works when consumers' own warehouses do the compute.
4. Governance: consumer sees app objects in their catalog; row access/column policies *inside the app code* segment by consumer tier — one published artifact, many entitlements.
5. The 2026 honest comparison: when a Delta Sharing share (open protocol) or a clean-room collaboration beats a Native App, and when the app wins (logic + data + UI in one package).

In [0]:
# Solution Q14 — Native Apps: ship logic+data, never ship the data

NATIVE_APP = """
1. THE TRUST MODEL (the trick that makes it work)
   - the app package contains application CODE plus REFERENCES to shared data
     (Snowflake objects). The consumer installs the app; it runs IN the
     consumer's account ON THE CONSUMER'S WAREHOUSE.
   - so: consumers pay their own compute (you never see their data, they never
     get your raw tables — only the app's functions/views), and the provider
     controls versions/revocation centrally. Data residency stays put.

2. VERSIONING & UPGRADES
   - app versions publish to the listing; consumers get upgrades through a
     channel (default/preview). Because consumers run YOUR code, a scoring-model
     or policy fix rolls out to every installation at once — impossible with a
     raw table share, where consumers write their own queries.

3. MONETIZATION
   - paid listings: flat or usage-based; the app can METER events through the
     built-in event table / reporting API. Consumer warehouses burn consumer
     credits for query execution; the listing bills for the data/logic product.

4. GOVERNANCE IN THE ARTIFACT
   - the app's own views/SPCs embed row access policies + entitlement logic:
     tier=free sees states, tier=pro sees counties, tier=enterprise sees ZIP —
     ONE artifact, per-consumer segmentation, zero per-tenant deployments.

5. WHEN NOT TO USE IT
   - consumer needs raw table access with their own BI -> Delta Sharing share
     (open, cross-platform, zero app packaging).
   - two parties must compute over combined sensitive data without exposing it
     -> clean room.
   - logic + curated data + controlled UX in one sellable unit -> Native App wins.
"""
print(NATIVE_APP)

# entitlement design drill — one artifact, tiered views
tiers = {"free": 1, "pro": 2, "enterprise": 3}
geo_grain = {1: "state", 2: "county", 3: "zip"}
assert geo_grain[tiers["enterprise"]] == "zip"
assert geo_grain[tiers["free"]] == "state"
print("Q14 OK: package = code + data references; consumer compute, provider control;")
print("     tiered policies inside one app; raw access -> sharing, joint secret data -> clean rooms")

## Q15 — Snowflake: Snowflake Intelligence & Semantic Views — Governed Agentic BI [HARD]

**Interviewer (Snowflake AI team):** Leadership wants "ask your data anything" chat over governed warehouse data. Walk me through Snowflake Intelligence (the 2025 agentic platform) and where a **semantic view** becomes non-negotiable.

**Requirements**
1. The stack: natural-language question -> agent plan -> governed objects -> grounded answer; what the agent is allowed to touch and what stops it from `SELECT *` on raw PII tables.
2. **Semantic views** (2025): SQL-defined business objects — metrics, dimensions, joins declared ONCE as metadata. Why NL-to-metric accuracy collapses without them (the agent shouldn't re-derive "active user" from columns every turn).
3. Governance hooks: row/role entitlements enforced through the semantic layer; the agent inherits the ASKING USER's permissions, not its own.
4. Comparison with the Databricks answer to the same problem (Genie/metric views) — what's genuinely similar (semantic layer + governed agent + evals) and what differs (platform objects, catalog depth).
5. The anti-pattern that kills agentic BI in production: free-text SQL on raw tables with no semantic layer — name the failure modes (metric drift, hallucinated joins, PII exposure) and the mitigation (curated semantic objects + tests + feedback loops).

In [0]:
# Solution Q15 — agentic BI: the semantic layer is the product

AGENTIC_BI = """
1. THE STACK
   question -> conversation agent -> PLAN (which semantic objects answer this?)
   -> governed query/analysis -> grounded answer + citations. The agent only
   reaches objects the governance model allows for the ASKING USER.

2. SEMANTIC VIEWS (the 2025 unlock)
   - SQL-defined objects declaring metrics, dimensions, and join paths as
     METADATA: one canonical 'active_users', 'ARR', 'churn' definition.
   - the agent SELECTS FROM the semantic view; it never re-derives a metric
     from raw columns. No semantic layer -> every NL turn is a fresh
     interpretation -> metric drift turn by turn.

3. GOVERNANCE HOOKS
   - semantic objects sit under role/row entitlements: the agent runs as the
     asking user; a sales analyst's question cannot see EU rows the analyst
     cannot see. PII columns never enter the plan at all.

4. VS THE DATABRIKS ANSWER
   - same shape: governed natural-language interface + semantic/metric layer
     (Genie spaces / metric views) + evals + feedback. Genuinely similar
     architecture; differences are platform objects (UC metric views + governed
     tags vs Snowflake semantic views) and catalog/lineage depth. In an
     interview, say the PATTERN twice and the product name once.

5. PRODUCTION ANTI-PATTERNS
   - free-text SQL over raw tables: metric drift (same word, two answers),
     hallucinated joins (invented FKs), PII exposure, unbounded scans.
   - mitigations: semantic objects as the ONLY surface, eval suites over the
     semantic layer (question -> expected object/metric), thumbs-down feedback
     routed to semantic owners, and PII policy tests in CI.
"""
print(AGENTIC_BI)

# the semantic-object routing drill — what the agent SHOULD select
objects = {
    "how many active users last week?":      "semantic.active_users",
    "ARR by segment for Q3":                 "semantic.arr_by_segment",
    "show me the raw user rows with emails": "BLOCKED (PII, not exposed via semantic layer)",
    "why did churn spike in October?":      "semantic.churn_timeseries + dimension drilldown",
}
for q, target in objects.items():
    print(f"   {q:42s} -> {target}")
assert objects["show me the raw user rows with emails"].startswith("BLOCKED")
assert all("semantic." in t for t in list(objects.values())[:2])
print("Q15 OK: NL agent + semantic views + user-scoped governance = the whole product;")
print("     raw-table free-text is the failure mode, on every platform")

## Q16 — Monitoring: LLM-as-Judge Evaluation — Agreement Illusion, Cohen's Kappa, Bias Mitigation [FAANG]

**Interviewer (Meta GenAI evals):** You're standing up an LLM-as-judge pipeline to grade model answers against human labels. The vendor dashboard reports "83% agreement with humans — production ready!" You have two hours to say why that number is nearly meaningless and what to report instead.

**Requirements**
1. The **agreement illusion**: when 85% of answers are genuinely GOOD, a judge that says GOOD almost always agrees ~83% — explain why raw agreement is dominated by prevalence.
2. Implement **Cohen's kappa from scratch** (`κ = (p_o − p_e)/(1 − p_e)`) and show the lenient judge scoring 83% agreement lands at κ≈0.2 — poor.
3. **Bias mitigation ladder**: detailed rubric, binary checklist, **position-swapped pairwise** evaluation — demonstrate each materially moves κ (here: 0.22 → 0.73).
4. Name the judge-specific biases to test for (position, verbosity/self-preference, leniency toward LLM text) and how a fixed human set + per-bias evals catches them.
5. State the production rule: report κ (and its CI), not agreement; gate on κ ≥ 0.6 for the decisions the judge actually makes.

In [0]:
# Solution Q16 — the agreement illusion, kappa from scratch, and the mitigation ladder
import random

random.seed(21)
N, P_GOOD = 400, 0.85     # ground truth: 85% of answers genuinely acceptable

def judge_labels(p_good_h1, p_good_h0):
    """P(judge says GOOD | human GOOD) and P(judge says GOOD | human BAD).
    A lenient judge upgrades bad answers: p_good_h0 high."""
    labels = []
    for _ in range(N):
        h = 1 if random.random() < P_GOOD else 0
        j = 1 if random.random() < (p_good_h1 if h == 1 else p_good_h0) else 0
        labels.append((h, j))
    return labels

def agreement(labels):
    return sum(h == j for h, j in labels) / len(labels)

def cohen_kappa(labels):
    n = len(labels)
    po = agreement(labels)
    ph = [sum(1 for h, _ in labels if h == k) / n for k in (0, 1)]
    pj = [sum(1 for _, j in labels if j == k) / n for k in (0, 1)]
    pe = ph[0] * pj[0] + ph[1] * pj[1]      # chance agreement from marginals
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

# 1) the lenient judge: upgrades 70% of BAD answers to GOOD
biased = judge_labels(p_good_h1=0.90, p_good_h0=0.70)
# 2) the mitigated judge: rubric + checklist + position-swapped pairwise
mitigated = judge_labels(p_good_h1=0.95, p_good_h0=0.15)

ag_b, k_b = agreement(biased), cohen_kappa(biased)
ag_m, k_m = agreement(mitigated), cohen_kappa(mitigated)
assert ag_b > 0.75, (ag_b, k_b)     # the dashboard number looks healthy
assert k_b < 0.40, (ag_b, k_b)      # kappa exposes the bias
assert k_m > k_b + 0.3 and k_m > 0.6, (k_b, k_m)   # mitigation ladder works

print(f"Q16a  lenient judge: raw agreement {ag_b:.0%}  <- what the vendor dashboard shows")
print(f"Q16b  ...but kappa = {k_b:.2f}  <- chance-corrected truth: POOR (gate is 0.6)")
print(f"Q16c  mitigated (rubric + checklist + position swap): kappa = {k_m:.2f}  -> usable")
print("Q16 OK: report kappa with a CI, never raw agreement; prevalence of GOOD (85%)")
print("     dominates raw agreement — a coin that says GOOD scores ~85% here")

## Q17 — Monitoring: Capacity Burn-Down Forecasting on Live Billing Data [HARD]

**Interviewer (Amazon FinOps):** The platform budget is 6,000 DBU/month. Finance wants to know **when we run out** — not "how much did we spend." You have `system.billing.usage`.

**Requirements**
1. Pull the last 28 days of daily DBU spend **from live system tables** and fit a linear trend (least squares by hand — no pandas).
2. Solve the cap-crossing day: when does the trend line hit `budget/30` per day?
3. Compute the **weekday profile** (avg DBU by day-of-week) — why seasonality matters before you trust a linear forecast, and how a weekday-only regression changes the answer.
4. State the alert design: burn-rate alerts vs threshold alerts, and why weekly seasonality demands week-over-week comparisons.
5. Follow-ups: per-workload attribution (tags/custom tags on `system.billing.usage`), and what to do when the trend says day 62 but a launch lands in 3 weeks.

In [0]:
# Solution Q17 — burn-down: live billing -> trend -> cap-crossing + weekday profile

rows = spark.sql("""
    SELECT usage_date AS d, SUM(usage_quantity) AS dbu
    FROM system.billing.usage
    WHERE usage_date >= current_date() - INTERVAL 28 DAYS AND usage_quantity > 0
    GROUP BY usage_date ORDER BY usage_date
""").collect()
n = len(rows)
assert n >= 15, n

xs, ys = list(range(n)), [float(r["dbu"]) for r in rows]
mx, my = sum(xs) / n, sum(ys) / n
slope = sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / sum((x - mx) ** 2 for x in xs)
intercept = my - slope * mx
total_28d, avg_d = sum(ys), sum(ys) / n

monthly_budget = 6000.0
daily_cap = monthly_budget / 30.0
t_star = (daily_cap - intercept) / slope          # when the trend crosses the cap
crossing = (f"day {t_star:.0f} ahead (trend still under cap)" if t_star > n
           else f"crossed cap day {t_star:.0f} (already burning above cap)")

weekday = spark.sql("""
    SELECT CASE dayofweek(usage_date) WHEN 1 THEN 'Sun' WHEN 2 THEN 'Mon' WHEN 3 THEN 'Tue'
           WHEN 4 THEN 'Wed' WHEN 5 THEN 'Thu' WHEN 6 THEN 'Fri' ELSE 'Sat' END AS dow,
           AVG(usage_quantity) AS avg_dbu
    FROM system.billing.usage
    WHERE usage_date >= current_date() - INTERVAL 28 DAYS AND usage_quantity > 0
    GROUP BY 1 ORDER BY min(usage_date)
""").collect()
dow_vals = [(r["dow"], round(float(r["avg_dbu"]), 1)) for r in weekday]

assert total_28d > 100 and len(dow_vals) >= 5
print(f"Q17   live system.billing.usage: {n} days, {total_28d:,.0f} DBU/28d, avg {avg_d:,.1f}/day")
print(f"Q17   linear trend {slope:+.1f} DBU/day per day; daily cap {daily_cap:.1f} -> {crossing}")
print(f"Q17   weekday profile: {dow_vals}")
print("Q17 OK: burn-down = trend vs cap + seasonality check; alert on burn RATE (slope),")
print("     not just absolute spend — thresholds fire too late for a launch spike")

## Q18 — Monitoring: RED + USE Observability Methods with OpenTelemetry Conventions [HARD]

**Interviewer (Uber observability):** The platform team says "we have dashboards." You say: "which method, which signals, which conventions?" They blink. Fix it.

**Requirements**
1. **RED** (for services/requests): Rate, Errors, Duration — compute all three per endpoint from a request log in one SQL pass, including p95 latency.
2. **USE** (for resources/pipelines): Utilization, Saturation, Errors — for the compute running the service (queue depth as the saturation proxy).
3. Say WHY you need both: RED answers "is the user hurting?" USE answers "which resource is the cause?" — debug RED through USE, not through more RED.
4. **OpenTelemetry semantic conventions**: name metrics canonically (`http.server.duration` histogram, `requests_total` counter, resource attributes) — why ad-hoc names (`api_latency_ms_v2`) murder cross-service aggregation and alert reuse.
5. Follow-up: map each signal to the Databricks stack (metrics tables, SQL alerts, Lakehouse Monitoring) and state the one number you'd put on a single slide for the exec review.

In [0]:
# Solution Q18 — RED per endpoint + USE per resource + OTel naming, one pass each
from pyspark.sql import functions as F

requests = spark.createDataFrame([
    # endpoint, ts, dur_ms, status, queue_depth, executor_util
    ("/api/orders",  1, 120, 200, 2, 0.55), ("/api/orders",  2, 180, 200, 3, 0.60),
    ("/api/orders",  3, 250, 500, 8, 0.92), ("/api/orders",  4, 300, 503, 9, 0.95),
    ("/api/search",  1, 400, 200, 5, 0.80), ("/api/search",  2, 900, 200, 12, 0.97),
    ("/api/search",  3, 700, 200, 10, 0.90),
], "endpoint string, ts int, dur_ms int, status int, queue int, util double")
requests.createOrReplaceTempView("requests")

# RED — is the user hurting? (per endpoint)
red = {r["endpoint"]: (r["rate"], float(r["error_ratio"]), r["p95_ms"]) for r in spark.sql("""
SELECT endpoint, COUNT(*) AS rate,
       ROUND(AVG(CASE WHEN status >= 500 THEN 1.0 ELSE 0 END), 3) AS error_ratio,
       percentile_approx(dur_ms, 0.95) AS p95_ms
FROM requests GROUP BY endpoint""").collect()}
assert red["/api/orders"] == (4, 0.5, 300), red
assert red["/api/search"] == (3, 0.0, 900), red

# USE — which resource is the cause? (for the compute)
use = spark.sql("""
SELECT MAX(queue) AS peak_queue_depth,           -- saturation proxy
       ROUND(MAX(util), 2) AS peak_util,          -- utilization
       SUM(CASE WHEN status >= 500 THEN 1 ELSE 0 END) AS errors
FROM requests""").collect()[0]
assert use["peak_queue_depth"] == 12 and use["peak_util"] == 0.97 and use["errors"] == 2

# OTel semantic conventions — names that compose across services
otel = {
    "R": "counter  requests_total{http.route='/api/orders'}",
    "E": "counter  http.server.errors... or ratio from requests_total labels",
    "D": "histogram http.server.duration -> p95 (server-side, standard name)",
    "U": "gauge    compute.utilization",
    "S": "gauge    queue.depth (pipeline saturation proxy)",
}
print(f"Q18   RED: {red}")
print(f"Q18   USE: queue {use['peak_queue_depth']}, util {use['peak_util']}, errors {use['errors']}")
for k, v in otel.items():
    print(f"      {k}: {v}")
print("Q18 OK: RED asks 'is the user hurting?', USE asks 'which resource is the cause?';")
print("     /api/search is SLOW (900ms p95) but HEALTHY (0 errors, queue < 10);")
print("     /api/orders FAILS 50% — that's the exec-slide number; OTel names make it")
print("     aggregable across every service, alert reuseable across every team")

# Day 23 Score Card

**Week 4 · Day 2 · FAANG / Top Product Company level — 8 [FAANG] + 10 [HARD]**

| Q | Topic | Tag | Solved (✓/✗) | First try? | Time (min) | Confidence (1–5) | Notes |
|---|---|---|---|---|---|---|---|
| Q1 | Sequential A/B testing — peeking, alpha spending, power | FAANG | ☐ | ☐ | ___ | ___ | correlated z-stats break fixed-α |
| Q2 | Account-takeover velocity features in SQL | FAANG | ☐ | ☐ | ___ | ___ | 24h device recency, not first_seen=ts |
| Q3 | KLL quantile sketch from scratch | FAANG | ☐ | ☐ | ___ | ___ | random compaction offset = unbiasedness |
| Q4 | Keyset pagination cursors vs OFFSET | FAANG | ☐ | ☐ | ___ | ___ | rows after cursor have GREATER tuples |
| Q5 | Geospatial grid join + haversine | FAANG | ☐ | ☐ | ___ | ___ | cell size bounds candidate fan-out |
| Q6 | Time-series interpolation / dense grids | HARD | ☐ | ☐ | ___ | ___ | last/next anchors; flat edges; ts IN trap |
| Q7 | Spark on Kubernetes | HARD | ☐ | ☐ | ___ | ___ | decommission migration = spot safety |
| Q8 | Serverless compute architecture | HARD | ☐ | ☐ | ___ | ___ | warm pools; locked fleet confs |
| Q9 | Paginated API ingestion + WAL + crash-resume | FAANG | ☐ | ☐ | ___ | ___ | WAL commit == unit of work |
| Q10 | JDBC incremental extraction | HARD | ☐ | ☐ | ___ | ___ | >= watermark + idempotent upsert |
| Q11 | ANSI strict mode + try_* family | HARD | ☐ | ☐ | ___ | ___ | strict core, tolerant edge |
| Q12 | Recommendation data model — implicit feedback | FAANG | ☐ | ☐ | ___ | ___ | impressions must be logged |
| Q13 | HTAP boundary — Lakebase + lakehouse | HARD | ☐ | ☐ | ___ | ___ | state vs history; branches ≠ backups |
| Q14 | Snowflake Native Apps + monetization | HARD | ☐ | ☐ | ___ | ___ | code+refs, consumer compute |
| Q15 | Snowflake Intelligence + semantic views | HARD | ☐ | ☐ | ___ | ___ | semantic layer is the product |
| Q16 | LLM-as-judge — kappa + bias mitigation | FAANG | ☐ | ☐ | ___ | ___ | report κ, never raw agreement |
| Q17 | Capacity burn-down on live billing | HARD | ☐ | ☐ | ___ | ___ | alert on burn rate + seasonality |
| Q18 | RED/USE + OTel conventions | HARD | ☐ | ☐ | ___ | ___ | RED = user, USE = cause |

**Debrief prompts**
1. Which questions exposed a gap between "I've heard of it" and "I can build it"?
2. Re-derive Q1's peeking inflation without notes — if you can't, you'll hand-wave it in a real interview.
3. Q4 and Q2 both hide a comparison-semantics trap. Can you state both traps in one sentence each?
4. Tomorrow (Day 24) candidates: streaming joins across watermarks vs d19's dual-watermark join, cost attribution per workload (vs d21 unit economics), Iceberg REST catalog deep dive (vs d18 matrix), multi-tenant catalog isolation at platform scale (vs d20 HHI), streaming lakehouse backfills, time-to-first-byte SLOs for data products.